# Image as a Discrete Signal: Mean, RMS, Correlation & K‑Means Segmentation (NumPy‑only)

**Goal:** Treat an **image** as a **discrete signal** (2D for grayscale, 3×2D for RGB) and connect
time-domain DSP ideas—**mean**, **RMS**, **correlation**—to an **unsupervised ML** operation (**K‑means**), implemented **from scratch** in NumPy.

**What you will do:**
1. Load an image (or use the built-in synthetic demo image).
2. Compute **mean** and **RMS** of the signal (per channel and overall).
3. Compute **channel correlations** (R↔G↔B).
4. Implement **K‑means clustering** in pure **NumPy** (no scikit‑learn).
5. **Quantize/segment** the image colors with **k clusters**.
6. Inspect **inertia (sum of squared error)**, and compare original vs segmented statistics.
7. (Optional) Use an **interactive slider** (if `ipywidgets` is available) to vary **k**.

> 💡 **DSP ↔ ML bridge:** K‑means minimizes the **sum of squared distances** (variance-like objective).  
> RMS/variance intuition → **why clusters are “prototypes”** of your signal’s values.

In [1]:
conda install numpy<2


Note: you may need to restart the kernel to use updated packages.


El sistema no puede encontrar el archivo especificado.


## 0) Setup & Imports

- Uses only **NumPy** + **Matplotlib**.
- Attempts to import `ipywidgets` for a slider; if not available (e.g., some Kaggle setups), the notebook will gracefully fall back to a manual function you can call with different `k`.

In [2]:
!pip install --user opencv-python

In [3]:
import matplotlib.pyplot as plt
import cv2
# (Optional, for Colab) -
#from google.colab import files
#uploaded = files.upload()  # choose .png kind of image.
#uploaded


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\luisn\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\luisn\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\luisn\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 736, in start
    self.io_loop.start()
  File "C:\Users\luisn\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

## 1) Load an Image (or Use Built‑in Synthetic Demo)

- If you have a local image, set `IMAGE_PATH` below (e.g., `'my_photo.jpg'`).
- If left as `None`, we'll generate a **synthetic** image with blocks and gradients so the segmentation is visually clear.
- Optionally, **downscale** large images to keep runtime snappy in Kaggle/Colab.

In [4]:
from PIL import Image
IMAGE_PATH = 'image.jpg'
img = Image.open(IMAGE_PATH).convert("RGB")

plt.imshow(img)
plt.axis("off")
plt.show()


NameError: name 'plt' is not defined

In [ ]:
import cv2
img_bgr = cv2.imread(filename)

# Convertir a RGB (para visualizar con matplotlib)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print("Shape (H,W,C):", img_rgb.shape)



In [ ]:
plt.imshow(img_rgb)
img=img_rgb

In [ ]:
import numpy as np
def mean_per_channel(img):
    # img: H x W x 3 in [0,1]
    return img.reshape(-1, img.shape[-1]).mean(axis=0)

def rms_per_channel(img):
    flat = img.reshape(-1, img.shape[-1])
    return np.sqrt((flat**2).mean(axis=0))

def overall_mean(img):
    return img.mean()

def overall_rms(img):
    return np.sqrt((img**2).mean())

mu_c = mean_per_channel(img)
rms_c = rms_per_channel(img)
mu_all = overall_mean(img)
rms_all = overall_rms(img)

print("Per-channel mean (R,G,B):", mu_c)
print("Per-channel RMS  (R,G,B):", rms_c)
print("Overall mean:", mu_all)
print("Overall RMS :", rms_all)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# Asegura escala [0,1] y tipo float
img_f = img.astype(np.float32)
if img_f.max() > 1.0:  # por si acaso venía en 0..255
    img_f = img_f / 255.0
H, W, C = img_f.shape
assert C == 3, "Se espera imagen RGB con 3 canales."

# Dataset de colores: N x 3
pixels = img_f.reshape(-1, 3)  # (H*W, 3)

# Elige K
K = 6  # cámbialo a tu gusto
kmeans = KMeans(n_clusters=K, n_init="auto")#, random_state=42)
labels = kmeans.fit_predict(pixels)
# Centroides en espacio RGB [0,1]
centers = kmeans.cluster_centers_  # shape (K, 3)
print(f'centroids {centers}')

In [ ]:
import pandas as pd
yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")   #

# Aplanamos todo
R = img_f[:,:,0].ravel()
G = img_f[:,:,1].ravel()
B = img_f[:,:,2].ravel()
X = xx.ravel()
Y = yy.ravel()

# DataFrame estilo dataset tabular
df = pd.DataFrame({
    "x": X.astype(np.int32),
    "y": Y.astype(np.int32),
    "r": R.astype(np.float32),
    "g": G.astype(np.float32),
    "b": B.astype(np.float32),
})

print("Shape del dataset:", df.shape)
df.sample(10)



In [ ]:
import matplotlib.pyplot as plt



In [ ]:
#kmeans.labels_
df['sklearn_kmeans']=kmeans.labels_
df

In [ ]:

def assign_clusters_dot(pixels, centroids):
    # Normalizamos a longitud 1 para que el dot dependa solo del ángulo
    pix_norm = pixels / np.linalg.norm(pixels, axis=1, keepdims=True)
    cent_norm = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)

    # Producto punto (N,K)
    sims = pix_norm @ cent_norm.T
    # Elegir el centroide con mayor similitud
    labels = np.argmax(sims, axis=1)
    return labels



In [ ]:

pixels = df[['r','g','b']].values
labels_dot = assign_clusters_dot(pixels, centers)

In [ ]:
K = 4  # número de clusters
# centroides aleatorios en [0,1] (R,G,B)
centroids = np.random.rand(K, 3)

print(f"Centroides iniciales aleatorios:\n {centroids}")


In [ ]:
swatch = np.stack([np.tile(c, (20, 100, 1)) for c in centroids], axis=0)
swatch = np.concatenate(list(swatch), axis=1)

plt.figure(figsize=(10,2))
plt.imshow(swatch)
plt.axis("off")
plt.title("Centroides iniciales (aleatorios en RGB)")
plt.show()


In [ ]:

pixels = df[['r','g','b']].values
labels_dot = assign_clusters_dot(pixels, centroids)

In [ ]:
df['labels_dot']=labels_dot

In [ ]:
df.describe()

In [ ]:
cent_norm = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
cent_norm

In [ ]:
np.linalg.norm(centroids, axis=1, keepdims=True),np.linalg.norm(cent_norm, axis=1, keepdims=True)

In [ ]:
pix_norm = pixels / np.linalg.norm(pixels, axis=1, keepdims=True)


In [ ]:
pixels=df[['r','g','b']].iloc[0].values  #ONE PIXEL
pix_norm = pixels / np.linalg.norm(pixels,  keepdims=True)
pix_norm

In [ ]:
dists = np.linalg.norm(cent_norm - pix_norm, axis=1)  # -> (K,)
cluster_num = dists.argmin()                          # -> int in [0..K-1]
print (dists)

In [ ]:
cluster_num=np.linalg.norm((pix_norm-cent_norm), axis=1, keepdims=True).argmin()
cluster_num

In [ ]:
# pixels: (N,3)
pixels = df[['r','g','b']].to_numpy(dtype=np.float32)   # N x 3
cent   = cent_norm.astype(np.float32)                   # K x 3
pix_norm = pixels / np.linalg.norm(pixels,  keepdims=True)

# Option A: straightforward broadcasting (creates an (N,K,3) temp)
diff  = pix_norm[:, None, :] - cent[None, :, :]           # N x K x 3
dists = np.linalg.norm(diff, axis=2)                    # N x K
labels = dists.argmin(axis=1)                           # N,
df['cluster'] = labels


In [ ]:
df.head()

In [ ]:



H, W, _ = img_f.shape
labels = df['labels_dot'].to_numpy()
labels_euc = df['cluster'].to_numpy()      # o df['labels_dot']
segmented = cent[labels].reshape(H, W, 3)  # centroides es tu arreglo de centroides (Kx3)
segmented_euc = cent[labels_euc].reshape(H, W, 3)  # centroides es tu arreglo de centroides (Kx3)

import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
plt.subplot(1,3,1)
plt.imshow(img_f)
plt.axis("off"); plt.title("Original")

plt.subplot(1,3,2)
plt.imshow(np.clip(segmented,0,1))
plt.axis("off"); plt.title("Imagen por dot similarity")

plt.subplot(1,3,3)
plt.imshow(np.clip(segmented_euc,0,1))
plt.axis("off"); plt.title("Imagen por euclidiano")
plt.show()

plt.show()


In [ ]:
cluster_id = 0   # el que quieras explorar
mask = (df['cluster'].to_numpy() == cluster_id).reshape(H, W)

plt.figure(figsize=(6,6))
plt.imshow(mask, cmap="gray")
plt.axis("off"); plt.title(f"Máscara cluster {cluster_id}")
plt.show()


In [ ]:
new_cents

In [ ]:
new_cents

In [ ]:
centroids

In [ ]:
new_cents = []
for cluster_id in range(K):
    # máscara booleana de los píxeles que caen en este cluster
    mask = (df['cluster'].to_numpy() == cluster_id)

    if np.any(mask):  # evitar clusters vacíos
        mean_rgb = df.loc[mask, ['r','g','b']].mean(axis=0).to_numpy()
    else:
        # si está vacío, re-inicializa con un píxel aleatorio
        mean_rgb = df[['r','g','b']].sample(1).to_numpy().ravel()

    new_cents.append(mean_rgb)

new_cents = np.vstack(new_cents)
print("Centroides recalculados:\n", new_cents)


In [ ]:
new_cent_norm = new_cents / np.linalg.norm(new_cents, axis=1, keepdims=True)
new_cent_norm


In [ ]:
df.head()

In [ ]:



H, W, _ = img_f.shape
labels = df['labels_dot'].to_numpy()
labels_euc = df['cluster'].to_numpy()      # o df['labels_dot']
segmented = cent[labels].reshape(H, W, 3)  # centroides es tu arreglo de centroides (Kx3)
segmented_euc = cent[labels_euc].reshape(H, W, 3)  # centroides es tu arreglo de centroides (Kx3)

import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
plt.subplot(1,3,1)
plt.imshow(img_f)
plt.axis("off"); plt.title("Original")

plt.subplot(1,3,2)
plt.imshow(np.clip(segmented,0,1))
plt.axis("off"); plt.title("Imagen por dot similarity")

plt.subplot(1,3,3)
plt.imshow(np.clip(segmented_euc,0,1))
plt.axis("off"); plt.title("Imagen por euclidiano")
plt.show()

plt.show()


In [ ]:



H, W, _ = img_f.shape
labels = df['labels_dot_2'].to_numpy()
labels_euc = df['cluster_2'].to_numpy()      # o df['labels_dot']
segmented2 = cent[labels].reshape(H, W, 3)  # centroides es tu arreglo de centroides (Kx3)
segmented_euc2 = cent[labels_euc].reshape(H, W, 3)  # centroides es tu arreglo de centroides (Kx3)

import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
plt.subplot(1,3,1)
plt.imshow(img_f)
plt.axis("off"); plt.title("Original")

plt.subplot(1,3,2)
plt.imshow(np.clip(segmented,0,1))
plt.axis("off"); plt.title("Imagen por dot similarity")

plt.subplot(1,3,3)
plt.imshow(np.clip(segmented2,0,1))
plt.axis("off"); plt.title("Imagen por similarity iter 2")
plt.show()

plt.show()


In [ ]:




import matplotlib.pyplot as plt
plt.figure(figsize=(12,6))
plt.subplot(1,3,1)
plt.imshow(img_f)
plt.axis("off"); plt.title("Original")

plt.subplot(1,3,2)
plt.imshow(np.clip(segmented_euc,0,1))
plt.axis("off"); plt.title("Imagen por dot eculidiano 1")

plt.subplot(1,3,3)
plt.imshow(np.clip(segmented_euc2,0,1))
plt.axis("off"); plt.title("Imagen por euclidiano t+1")
plt.show()

plt.show()


In [ ]:
df

## 2) Signal Measurements: Mean & RMS

We treat the image as a **discrete signal** of pixels. For RGB, think of **three signals** sharing the same time/space index.

- **Mean** (DC level / brightness):  
  \u03BC = average of pixel values.

- **RMS** (energy-like amplitude):  
  \u221A(E[x²]) — sensitive to both **level** and **variation**.

We compute per-channel and overall metrics.

## 3) Channel Correlation (R↔G↔B)

Compute the **Pearson correlation** between channels (R, G, B).  
Intuition: Highly correlated channels indicate similar color dynamics (e.g., green↔blue in ocean/forest images).